In [31]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import nibabel as nib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

DATASET_DIR = "/scratch/users/serami/229data/UCSF-PDGM-v5"
METADATA_CSV = "/scratch/users/serami/229data/UCSF-PDGM-metadata_v5.csv"

GRADE_COLUMN = "WHO CNS Grade"
PATIENT_ID_COLUMN = "ID"

# modalities to use
MODALITIES = [
    "T1c",
    "FLAIR",
    "ADC",
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_nifti(path):
    """
    Load NIfTI image and return numpy array.
    """
    nii = nib.load(path)
    data = nii.get_fdata()
    return data


def extract_basic_features(image, mask):
    """
    Extract simple statistical features inside tumor mask.
    """

    masked = image[mask > 0]

    # remove invalid values
    masked = masked[np.isfinite(masked)]

    if len(masked) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "median": np.nan,
            "p10": np.nan,
            "p90": np.nan,
        }

    return {
        "mean": np.mean(masked),
        "std": np.std(masked),
        "min": np.min(masked),
        "max": np.max(masked),
        "median": np.median(masked),
        "p10": np.percentile(masked, 10),
        "p90": np.percentile(masked, 90),
    }


def tumor_shape_features(mask):
    """
    Simple shape features.
    """

    tumor_voxels = np.sum(mask > 0)

    return {
        "tumor_volume_voxels": tumor_voxels,
    }


def find_scan_file(patient_dir, keyword):
    """
    Find file containing keyword.
    Case-insensitive.
    """

    files = glob.glob(os.path.join(patient_dir, "*.nii.gz"))

    for f in files:

        filename = os.path.basename(f).lower()

        if keyword.lower() in filename:
            return f

    return None


# ============================================================
# LOAD METADATA
# ============================================================

metadata = pd.read_csv(METADATA_CSV)

# clean ID column
metadata[PATIENT_ID_COLUMN] = (
    metadata[PATIENT_ID_COLUMN]
    .astype(str)
    .str.strip()
)

# remove missing grades
metadata = metadata.dropna(subset=[GRADE_COLUMN])

print(f"Metadata patients: {len(metadata)}")

# ============================================================
# GET PATIENT FOLDERS
# ============================================================

patient_dirs = sorted(
    glob.glob(os.path.join(DATASET_DIR, "UCSF-PDGM-*_nifti"))
)

print(f"Found patient folders: {len(patient_dirs)}")

# ============================================================
# FEATURE EXTRACTION
# ============================================================

all_features = []
all_labels = []
all_patient_ids = []

for patient_dir in patient_dirs:

    folder_name = os.path.basename(patient_dir)

    # IMPORTANT:
    # keep _nifti because CSV IDs apparently include it
    patient_id = folder_name.replace("_nifti", "").strip()
    # --------------------------------------------------------
    # match metadata
    # --------------------------------------------------------

    # --------------------------------------------------------
# normalize patient number
# --------------------------------------------------------

    # folder:
    # UCSF-PDGM-0477 -> 477
    patient_num = int(patient_id.split("-")[-1])

    # metadata:
    # UCSF-PDGM-477 -> 477
    metadata_nums = (
        metadata[PATIENT_ID_COLUMN]
        .astype(str)
        .str.extract(r'(\d+)')[0]
        .astype(int)
    )

    # match
    row = metadata[
        metadata_nums == patient_num
    ]

    if len(row) == 0:
        continue

    grade = row.iloc[0][GRADE_COLUMN]

    # --------------------------------------------------------
    # segmentation file
    # --------------------------------------------------------

    seg_file = (
        find_scan_file(patient_dir, "tumor")
        or find_scan_file(patient_dir, "seg")
        or find_scan_file(patient_dir, "mask")
    )

    if seg_file is None:
        print(f"No segmentation found for {patient_id}")
        continue

    print(f"Segmentation: {os.path.basename(seg_file)}")

    # --------------------------------------------------------
    # load segmentation
    # --------------------------------------------------------

    try:
        mask = load_nifti(seg_file)
    except Exception as e:
        print(f"Failed loading segmentation: {e}")
        continue

    patient_features = {}

    # --------------------------------------------------------
    # shape features
    # --------------------------------------------------------

    shape_feats = tumor_shape_features(mask)

    for k, v in shape_feats.items():
        patient_features[k] = v

    # --------------------------------------------------------
    # modality features
    # --------------------------------------------------------

    for modality in MODALITIES:

        scan_file = find_scan_file(patient_dir, modality)

        if scan_file is None:

            print(f"Missing {modality}")

            for feat_name in [
                "mean",
                "std",
                "min",
                "max",
                "median",
                "p10",
                "p90",
            ]:
                patient_features[f"{modality}_{feat_name}"] = np.nan

            continue

        print(f"{modality}: {os.path.basename(scan_file)}")

        try:

            image = load_nifti(scan_file)

            # shape safety check
            if image.shape != mask.shape:
                print(f"Shape mismatch for {modality}")
                continue

            feats = extract_basic_features(image, mask)

            for k, v in feats.items():
                patient_features[f"{modality}_{k}"] = v

        except Exception as e:
            print(f"Error processing {modality}: {e}")

    # --------------------------------------------------------
    # save patient
    # --------------------------------------------------------

    all_features.append(patient_features)
    all_labels.append(grade)
    all_patient_ids.append(patient_id)

    print(f"Added patient: {patient_id}")

# ============================================================
# BUILD FEATURE MATRIX
# ============================================================

print("\n==============================")
print("DATA SUMMARY")
print("==============================")

print(f"Patients processed: {len(all_features)}")

if len(all_features) == 0:
    raise ValueError("No patients processed.")

X = pd.DataFrame(all_features)

print("Feature matrix shape:", X.shape)

# ============================================================
# IMPUTE MISSING VALUES
# ============================================================

imputer = SimpleImputer(strategy="median")

X_imputed = imputer.fit_transform(X)

# ============================================================
# ENCODE LABELS
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(all_labels)

print("\nClasses:")

for i, c in enumerate(label_encoder.classes_):
    print(i, "->", c)

# ============================================================
# TRAIN / VAL / TEST SPLIT
# ============================================================

# first split:
# 80% train+val
# 20% test

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# second split:
# from remaining 80%:
# 75% train
# 25% val
#
# overall:
# 60% train
# 20% val
# 20% test

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval,
)

print("Train size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# ============================================================
# RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
)

rf.fit(X_train, y_train)

# ============================================================
# EVALUATION
# ============================================================

y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("\n==============================")
print("RESULTS")
print("==============================")

print(f"\nAccuracy: {acc:.4f}")

print("\nClassification Report:")

print(
    classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_.astype(str),
    zero_division=0,
)
)

print("\nConfusion Matrix:")

print(confusion_matrix(y_test, y_pred))

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_,
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False,
)

print("\nTop 20 Features:")
print(feature_importance.head(20))

# save
feature_importance.to_csv(
    "random_forest_feature_importance.csv",
    index=False,
)

print("\nSaved feature importance CSV.")

Metadata patients: 501
Found patient folders: 65
No segmentation found for UCSF-PDGM-0477
No segmentation found for UCSF-PDGM-0478
No segmentation found for UCSF-PDGM-0479
No segmentation found for UCSF-PDGM-0480
No segmentation found for UCSF-PDGM-0481
No segmentation found for UCSF-PDGM-0482
No segmentation found for UCSF-PDGM-0483
Segmentation: UCSF-PDGM-0484_tumor_segmentation.nii.gz
T1c: UCSF-PDGM-0484_T1c.nii.gz
FLAIR: UCSF-PDGM-0484_FLAIR.nii.gz
ADC: UCSF-PDGM-0484_ADC.nii.gz
Added patient: UCSF-PDGM-0484
Segmentation: UCSF-PDGM-0485_tumor_segmentation.nii.gz
T1c: UCSF-PDGM-0485_T1c_bias.nii.gz
FLAIR: UCSF-PDGM-0485_FLAIR.nii.gz
ADC: UCSF-PDGM-0485_ADC.nii.gz
Added patient: UCSF-PDGM-0485
Segmentation: UCSF-PDGM-0486_tumor_segmentation.nii.gz
T1c: UCSF-PDGM-0486_T1c_bias.nii.gz
FLAIR: UCSF-PDGM-0486_FLAIR_bias.nii.gz
ADC: UCSF-PDGM-0486_ADC.nii.gz
Added patient: UCSF-PDGM-0486
Segmentation: UCSF-PDGM-0487_tumor_segmentation.nii.gz
T1c: UCSF-PDGM-0487_T1c_bias.nii.gz
FLAIR: UCSF-

In [33]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
)

# ------------------------------------------------------------
# TEST SET
# ------------------------------------------------------------

y_test_pred = rf.predict(X_test)
y_test_prob = rf.predict_proba(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
)

test_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_balanced_acc = balanced_accuracy_score(
    y_test,
    y_test_pred,
)

test_auc = roc_auc_score(
    y_test,
    y_test_prob,
    multi_class="ovr",
    labels=np.arange(len(label_encoder.classes_)),
)

# ============================================================
# PRINT TEST RESULTS
# ============================================================

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(f"Accuracy:            {test_accuracy:.4f}")
print(f"Macro F1:            {test_macro_f1:.4f}")
print(f"Weighted F1:         {test_weighted_f1:.4f}")
print(f"Macro Precision:     {test_precision:.4f}")
print(f"Macro Recall:        {test_recall:.4f}")
print(f"Balanced Accuracy:   {test_balanced_acc:.4f}")
print(f"Multiclass ROC-AUC:  {test_auc:.4f}")

# ============================================================
# TEST CLASSIFICATION REPORT
# ============================================================

print("\nTest Classification Report:")

print(
    classification_report(
        y_test,
        y_test_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_.astype(str),
        zero_division=0,
    )
)

# ============================================================
# TEST CONFUSION MATRIX
# ============================================================

print("\nTest Confusion Matrix:")

print(confusion_matrix(y_test, y_test_pred))


TEST RESULTS
Accuracy:            0.9167
Macro F1:            0.8095
Weighted F1:         0.9048
Macro Precision:     0.9545
Macro Recall:        0.7500
Balanced Accuracy:   0.7500
Multiclass ROC-AUC:  nan

Test Classification Report:
              precision    recall  f1-score   support

           2       0.00      0.00      0.00         0
           3       1.00      0.50      0.67         2
           4       0.91      1.00      0.95        10

    accuracy                           0.92        12
   macro avg       0.64      0.50      0.54        12
weighted avg       0.92      0.92      0.90        12


Test Confusion Matrix:
[[ 1  1]
 [ 0 10]]
